# LongFlow — score the quality night on GPU (with the EAR PACK)

Runtime: **any GPU**. Reads `quality_eval.zip` from Drive root. Reports:
(1) held-out WER/sim curves per checkpoint per arm (overfit watch);
(2) EAR-PACK curves on the four 30s-chunked renders + the pre-registered
DATA PAYS / SIZE PAYS / WIN verdicts (NOTES "QUALITY NIGHT").


In [ ]:
# ===== COLD START — run me first, wait for READY =====
NOTEBOOK_VERSION = "Score quality night (GPU) v1.0 (2026-08-19)"
print(f"*** {NOTEBOOK_VERSION} ***")
!pip install -q faster-whisper jiwer whisper-normalizer speechbrain praat-parselmouth

import torch
assert torch.cuda.is_available(), "no GPU — pick a GPU runtime"
import glob, json, os, sys, zipfile
import numpy as np
import soundfile as sf

if os.path.exists("/content/LongFlow/src"):
    !cd /content/LongFlow && git pull -q
else:
    !git clone -q https://github.com/Josh-E-S/LongFlow.git /content/LongFlow || true
sys.path.insert(0, "/content/LongFlow")
!cd /content/LongFlow && git log --oneline -1
from src.eval.metrics import clip_metrics, ear_pack

from google.colab import drive
drive.mount("/content/drive")
candidates = glob.glob("/content/drive/MyDrive/quality_eval*.zip")  # root only
assert candidates, "no quality_eval*.zip in Drive root"
zip_path = candidates[0]
print(f"using {zip_path} ({os.path.getsize(zip_path)/1e9:.2f} GB)")
AUD = "/content/eval_audio"
os.makedirs(AUD, exist_ok=True)
with zipfile.ZipFile(zip_path) as z:
    z.extractall(AUD)
print(f"extracted {len(os.listdir(AUD))} entries")
print("READY")


In [ ]:
# ===== Part 1: held-out curves per checkpoint per arm (overfit watch) =====
with open(f"{AUD}/manifest.json") as f:
    manifest = json.load(f)
results = {"held_out": {}, "chunked": {}, "verdicts": {}}

for key in sorted(manifest["checkpoints"], key=lambda k: (int(k.split(":")[0]), k)):
    entries = manifest["checkpoints"][key]
    rows = []
    for e in entries:
        m = clip_metrics(f"{AUD}/{e['audio']}", e["text"],
                         f"{AUD}/{e['teacher_audio']}", device="cuda")
        rows.append({**m, "utt_id": e["utt_id"], "target_words": e["target_words"]})
    wers = sorted(r["wer"] for r in rows)
    sims = sorted(r["speaker_sim"] for r in rows)
    confs = sorted(r["asr_confidence"] for r in rows)
    results["held_out"][key] = {
        "n": len(rows), "wer_median": wers[len(wers) // 2],
        "sim_median": sims[len(sims) // 2],
        "asr_conf_median": confs[len(confs) // 2], "rows": rows,
    }
    o = results["held_out"][key]
    print(f"{key:>15}: n={o['n']:>3}  wer_med={o['wer_median']:.3f}  "
          f"sim_med={o['sim_median']:.3f}  conf={o['asr_conf_median']:.2f}")

for arm in ("v3_640", "v3_960"):
    steps = sorted(int(k.split(":")[0]) for k in results["held_out"] if k.endswith(arm))
    if len(steps) >= 2:
        best = min(steps, key=lambda s: results["held_out"][f"{s}:{arm}"]["wer_median"])
        if best != steps[-1]:
            print(f"WATCH {arm}: best held-out WER at step {best}, not {steps[-1]} — "
                  f"E3 overfit signature; use {best} as the operating checkpoint")


In [ ]:
# ===== Part 2: EAR PACK on the 30s-chunked renders + verdicts =====
CHUNK_S = 30.0  # nominal; early window = first 15s of each ~30s chunk
TEACHER_EARLY_HNR = 13.88  # forensics 2026-08-19
CLEANABL_REF = 9.93        # cleanabl_p2 early HNR (closest incumbent measurement)

curves = {}
for p in sorted(glob.glob(f"{AUD}/closed_loop/qc_*.wav")):
    tag = os.path.basename(p)[:-4]
    print(f"ear-packing {tag}...", flush=True)
    curves[tag] = ear_pack(p, win_s=2.0, hop_s=1.0)
    ep = curves[tag]
    results["chunked"][tag] = {k: v for k, v in ep.items() if k.endswith("_median")}
    print(f"  {tag}: HNR_med={ep['hnr_median']:.2f}  CPPS_med={ep['cpps_median']:.2f}  "
          f"flat_med={ep['flatness_median']:.4f}  shimmer_med={ep['shimmer_median']:.4f}")

# golden-window (per-chunk early) HNR: approximate chunk starts every ~CHUNK_S
def early_hnr(tag):
    ep = curves[tag]
    t = np.array(ep["t"]); h = np.array(ep["hnr"], dtype=float)
    rel = np.mod(t, CHUNK_S)
    m = (rel < 15) & np.isfinite(h)
    return float(np.median(h[m])) if m.any() else float("nan")

for tag in list(curves):
    e = early_hnr(tag)
    results["chunked"][tag]["early_hnr"] = e
    print(f"{tag}: golden-window (early) HNR = {e:.2f}")

t_ref = results["chunked"].get("qc_teacher", {}).get("early_hnr", TEACHER_EARLY_HNR)
c_ref = results["chunked"].get("qc_cleanabl", {}).get("early_hnr", CLEANABL_REF)
h640 = results["chunked"].get("qc_640", {}).get("early_hnr", float("nan"))
h960 = results["chunked"].get("qc_960", {}).get("early_hnr", float("nan"))

v = []
if np.isfinite(h640):
    v.append(f"DATA {'PAYS' if h640 >= c_ref + 1.0 else 'did not pay'} "
             f"(640-on-v3 {h640:.2f} vs cleanabl {c_ref:.2f}, bar +1.0)")
if np.isfinite(h960) and np.isfinite(h640):
    v.append(f"SIZE {'PAYS' if h960 >= h640 + 1.0 else 'did not pay'} "
             f"(960 {h960:.2f} vs 640 {h640:.2f}, bar +1.0)")
best = max([x for x in (h640, h960) if np.isfinite(x)], default=float("nan"))
if np.isfinite(best):
    halfway = c_ref + (t_ref - c_ref) / 2 if t_ref > c_ref else t_ref
    v.append(f"QUALITY-NIGHT {'WIN (metric half — Josh ear decides)' if best >= halfway else 'below the halfway bar'} "
             f"(best {best:.2f} vs halfway {halfway:.2f} toward teacher {t_ref:.2f})")
results["verdicts"]["quality_night"] = v
print("\nVERDICTS:")
for line in v:
    print(" ", line)

with open("/content/drive/MyDrive/quality_metrics.json", "w") as f:
    json.dump(results, f, indent=2)
print("\nmetrics on Drive root: quality_metrics.json")
print("Listening (never skipped): qc_teacher then the best student render, full pass.")


In [ ]:
# ===== Part 3: THE GRAPHS — ear-pack curves, inline + saved to Drive =====
import matplotlib.pyplot as plt

COLORS = {"qc_teacher": "k", "qc_cleanabl": "tab:cyan",
          "qc_640": "tab:red", "qc_960": "tab:blue"}

def within_chunk_curve(tag, key, chunk_s=30.0, bin_s=3.0):
    ep = curves[tag]
    t = np.array(ep["t"]); v = np.array(ep[key], dtype=float)
    rel = np.mod(t, chunk_s)
    bins = np.arange(0, chunk_s + bin_s, bin_s)
    med = []
    for j in range(len(bins) - 1):
        m = (rel >= bins[j]) & (rel < bins[j + 1]) & np.isfinite(v)
        med.append(np.median(v[m]) if m.any() else np.nan)
    return bins[:-1] + bin_s / 2, np.array(med)

# Figure 1: within-chunk decay, all four engines
metrics_plot = [("hnr", "HNR dB (voice cleanliness)"),
                ("cpps", "CPPS (breathiness, higher=cleaner)"),
                ("flatness", "Spectral flatness (noisiness)"),
                ("shimmer", "Shimmer (amplitude instability)")]
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for ax, (mk, label) in zip(axes.flat, metrics_plot):
    for tag in sorted(curves):
        t, v = within_chunk_curve(tag, mk)
        ax.plot(t, v, color=COLORS.get(tag, "gray"), label=tag.replace("qc_", ""),
                lw=2 if tag in ("qc_teacher", "qc_640", "qc_960") else 1.2)
    ax.set_title(label); ax.set_xlabel("seconds since chunk start"); ax.grid(alpha=0.3)
axes[0, 0].legend(fontsize=9)
fig.suptitle("Quality night — within-chunk ear-pack curves (30s chunks)")
fig.tight_layout()
fig.savefig("/content/drive/MyDrive/quality_fig1_earpack_decay.png", dpi=110)
plt.show()

# Figure 2: full-render HNR timelines
fig, ax = plt.subplots(figsize=(14, 4.5))
for tag in sorted(curves):
    ep = curves[tag]
    ax.plot(ep["t"], ep["hnr"], color=COLORS.get(tag, "gray"),
            label=tag.replace("qc_", ""), lw=1, alpha=0.85)
ax.set_xlabel("seconds"); ax.set_ylabel("HNR dB"); ax.grid(alpha=0.3); ax.legend(fontsize=9)
ax.set_title("Full-render HNR timelines — the golden-window floor, all engines")
fig.tight_layout()
fig.savefig("/content/drive/MyDrive/quality_fig2_hnr_timeline.png", dpi=110)
plt.show()

# Figure 3: held-out WER curve per checkpoint (the overfit watch, visually)
steps_by_arm = {}
for key, o in results["held_out"].items():
    step, arm = key.split(":")
    steps_by_arm.setdefault(arm, []).append((int(step), o["wer_median"]))
fig, ax = plt.subplots(figsize=(8, 4.5))
for arm, pts in sorted(steps_by_arm.items()):
    pts.sort()
    ax.plot([p[0] for p in pts], [p[1] for p in pts], marker="o",
            label=arm, color={"v3_640": "tab:red", "v3_960": "tab:blue"}.get(arm, "gray"))
ax.set_xlabel("training step"); ax.set_ylabel("held-out WER (median)")
ax.grid(alpha=0.3); ax.legend()
ax.set_title("Held-out WER vs steps — best point = operating checkpoint (E3 watch)")
fig.tight_layout()
fig.savefig("/content/drive/MyDrive/quality_fig3_wer_curve.png", dpi=110)
plt.show()
print("figures saved to Drive root: quality_fig1/2/3_*.png")
